# 02 — Pré-processamento

**Dataset:** Vehicle Collision Data in Seattle (2005–2019)

**Tarefa:** Tratamento estrutural e padronização dos dados
## Estrutura deste notebook

    1. Carregamento
    2. Extrair features temporais de DATE/TIME: hora, mês, dia da semana
    3. Remover colunas irrelevantes
    4. Remover leakage
    5. Remover multicolinearidade
    6. Remover colunas com > 50% nulos
    7. Remover 19 linhas duplicadas
    8. Imputar SNOW e SNWD com 0
    9. Imputar demais nulos com mediana (numéricas) ou moda (categóricas)
    10. Tratar outliers
    11. Converter booleanas para int (0/1)
    12. One-Hot Encoding (antes de remover)
    13. LIGHTCOND manter como ordinal inteiro
    14. Split 80/20 estratificado por SEVERITYCODE (antes do scaler)
    15. StandardScaler nas numéricas contínuas — fit só no treino, transform em treino e teste
    16. Verificar shape final de X_train, X_test, y_train, y_test
    17. Salvar X_train, X_test, y_train, y_test em arquivos (ex: .pkl ou .csv)
  


## 1. Carregamento

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import joblib

df = pd.read_csv('/content/seattle_collision_data_2005_2019.csv')
print("Shape inicial do dataset:", df.shape)

Shape inicial do dataset: (111882, 34)


## 2. Extrair features temporais de DATE/TIME: hora, mês, dia da semana (antes de remover)

In [2]:
df['DATE'] = pd.to_datetime(df['DATE'])
df['TIME'] = pd.to_datetime(df['TIME'], errors='coerce')

df['mes'] = df['DATE'].dt.month
df['dia_da_semana'] = df['DATE'].dt.dayofweek
df['hora'] = df['TIME'].dt.hour

df['hora'] = df['hora'].fillna(df['hora'].median()).astype(int)

print("2. Features temporais extraídas!")
display(df[['DATE', 'TIME', 'mes', 'dia_da_semana', 'hora']].head())

2. Features temporais extraídas!


,DATE,TIME,mes,dia_da_semana,hora
0,2005-01-10,1970-01-01 00:00:00.000000002,1,0,0
1,2005-01-10,1970-01-01 00:00:00.000000007,1,0,0
2,2005-01-10,1970-01-01 00:00:00.000000007,1,0,0
3,2005-01-10,1970-01-01 00:00:00.000000010,1,0,0
4,2005-01-10,1970-01-01 00:00:00.000000010,1,0,0


## 3. Remover colunas irrelevantes: Unnamed: 0, SPDCASENO, DATE, TIME

In [3]:
df.drop(columns=['Unnamed: 0', 'SPDCASENO', 'DATE', 'TIME'], inplace=True, errors='ignore')

print("3. Colunas irrelevantes removidas!")
print("Shape atual:", df.shape)

3. Colunas irrelevantes removidas!
Shape atual: (111882, 33)


## 4. Remover leakage: INJURIES, SERIOUSINJURIES, FATALITIES

In [4]:
df.drop(columns=['INJURIES', 'SERIOUSINJURIES', 'FATALITIES'], inplace=True, errors='ignore')

print("4. Colunas de leakage removidas!")
print("Shape atual:", df.shape)

4. Colunas de leakage removidas!
Shape atual: (111882, 30)


## 5. Remover multicolinearidade: TMAX, TMIN, WSF5

In [5]:
df.drop(columns=['TMAX', 'TMIN', 'WSF5'], inplace=True, errors='ignore')

print("5. Colunas multicolineares removidas!")
print("Shape atual:", df.shape)

5. Colunas multicolineares removidas!
Shape atual: (111882, 27)


## 6. Remover colunas com > 50% nulos: response_type, response_time

In [6]:
df.drop(columns=['response_type', 'response_time'], inplace=True, errors='ignore')

print("6. Colunas com muitos nulos removidas!")
print("Shape atual:", df.shape)

6. Colunas com muitos nulos removidas!
Shape atual: (111882, 25)


## 7. Remover 19 linhas duplicadas

In [7]:
linhas_antes = len(df)
df.drop_duplicates(inplace=True)
linhas_depois = len(df)

print(f"7. Linhas duplicadas removidas: {linhas_antes - linhas_depois}")
print("Shape atual:", df.shape)

7. Linhas duplicadas removidas: 46
Shape atual: (111836, 25)


## 8. Imputar SNOW e SNWD com 0

In [8]:
df['SNOW'] = df['SNOW'].fillna(0)
df['SNWD'] = df['SNWD'].fillna(0)

print("8. Valores nulos de SNOW e SNWD substituídos por 0!")
print("Valores nulos em SNOW:", df['SNOW'].isnull().sum())
print("Valores nulos em SNWD:", df['SNWD'].isnull().sum())

8. Valores nulos de SNOW e SNWD substituídos por 0!
Valores nulos em SNOW: 0
Valores nulos em SNWD: 0


## 9. Imputar demais nulos com mediana (numéricas) ou moda (categóricas)

In [9]:
num_cols = df.select_dtypes(include=['int64', 'float64']).columns
for col in num_cols:
    df[col] = df[col].fillna(df[col].median())

cat_cols = df.select_dtypes(include=['object', 'category']).columns
for col in cat_cols:
    df[col] = df[col].fillna(df[col].mode()[0])

print("9. Imputação finalizada!")
print("Total de valores nulos no dataset inteiro agora:", df.isnull().sum().sum())

9. Imputação finalizada!
Total de valores nulos no dataset inteiro agora: 0


## 10. Tratar outliers em PERSONCOUNT, PEDCOUNT, PEDCYLCOUNT, VEHCOUNT

In [10]:
cols_outliers = ['PERSONCOUNT', 'PEDCOUNT', 'PEDCYLCOUNT', 'VEHCOUNT']

print("10. Tratamento de Outliers (Abordagem por Percentil Seguro):")
for col in cols_outliers:
    limite_superior = df[col].quantile(0.99)

    if limite_superior == 0:
        limite_superior = 1.0

    df[col] = np.where(df[col] > limite_superior, limite_superior, df[col])
    print(f"-> Coluna '{col}': Valores limitados ao teto de {limite_superior}")

print("\nValidação - Valores máximos atuais (devem ser maiores que 0):")
display(df[cols_outliers].describe().loc[['max']])

10. Tratamento de Outliers (Abordagem por Percentil Seguro):
-> Coluna 'PERSONCOUNT': Valores limitados ao teto de 7.0
-> Coluna 'PEDCOUNT': Valores limitados ao teto de 1.0
-> Coluna 'PEDCYLCOUNT': Valores limitados ao teto de 1.0
-> Coluna 'VEHCOUNT': Valores limitados ao teto de 4.0

Validação - Valores máximos atuais (devem ser maiores que 0):


,PERSONCOUNT,PEDCOUNT,PEDCYLCOUNT,VEHCOUNT
max,7.0,1.0,1.0,4.0


## 11. Converter booleanas para int (0/1)

In [11]:
bool_cols = df.select_dtypes(include=['bool']).columns
df[bool_cols] = df[bool_cols].astype(int)

print("11. Booleanas originais convertidas para inteiros com sucesso!")

11. Booleanas originais convertidas para inteiros com sucesso!


## 12. One-Hot Encoding: COLLISIONTYPE, WEATHER, ROADCOND, JUNCTIONTYPE

In [12]:
cols_to_encode = ['COLLISIONTYPE', 'WEATHER', 'ROADCOND', 'JUNCTIONTYPE']
df = pd.get_dummies(df, columns=cols_to_encode, drop_first=True)
bool_cols = df.select_dtypes(include=['bool']).columns
df[bool_cols] = df[bool_cols].astype(int)

print("12. One-Hot Encoding aplicado e convertido para inteiros!")
df.info()

12. One-Hot Encoding aplicado e convertido para inteiros!
<class 'pandas.core.frame.DataFrame'>
Index: 111836 entries, 0 to 111881
Data columns (total 45 columns):
 #   Column                            Non-Null Count   Dtype  
---  ------                            --------------   -----  
 0   longitude                         111836 non-null  float64
 1   latitude                          111836 non-null  float64
 2   SEVERITYCODE                      111836 non-null  int64  
 3   PERSONCOUNT                       111836 non-null  float64
 4   PEDCOUNT                          111836 non-null  float64
 5   PEDCYLCOUNT                       111836 non-null  float64
 6   VEHCOUNT                          111836 non-null  float64
 7   INATTENTIONIND                    111836 non-null  int64  
 8   UNDERINFL                         111836 non-null  int64  
 9   LIGHTCOND                         111836 non-null  int64  
 10  SPEEDING                          111836 non-null  int64  
 11 

## 13. LIGHTCOND manter como ordinal inteiro

In [13]:
mapa_luz = {
    'Daylight': 5,
    'Dawn': 4,
    'Dusk': 3,
    'Dark - Street Lights On': 2,
    'Dark - No Street Lights': 1,
    'Dark - Unknown Lighting': 0,
    'Other': 0,
    'Unknown': 0
}
df['LIGHTCOND'] = df['LIGHTCOND'].map(mapa_luz).fillna(0).astype(int)

print("13. LIGHTCOND convertida para ordinal inteiro com sucesso!")
display(df['LIGHTCOND'].value_counts())

13. LIGHTCOND convertida para ordinal inteiro com sucesso!


,count
LIGHTCOND,
0,111836


## 14. Split 80/20 estratificado por SEVERITYCODE (antes do scaler)

In [14]:
X = df.drop('SEVERITYCODE', axis=1)
y = df['SEVERITYCODE']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

print("13. Divisão Treino/Teste concluída!")
print("Proporção das classes no Treino:\n", y_train.value_counts(normalize=True))
print("Proporção das classes no Teste:\n", y_test.value_counts(normalize=True))

13. Divisão Treino/Teste concluída!
Proporção das classes no Treino:
 SEVERITYCODE
0    0.643303
1    0.337931
2    0.017068
3    0.001699
Name: proportion, dtype: float64
Proporção das classes no Teste:
 SEVERITYCODE
0    0.643285
1    0.337938
2    0.017078
3    0.001699
Name: proportion, dtype: float64


## 15. StandardScaler nas numéricas contínuas — fit só no treino, transform em treino e teste

In [15]:
cont_cols = ['PERSONCOUNT', 'PEDCOUNT', 'PEDCYLCOUNT', 'VEHCOUNT']
scaler = StandardScaler()

X_train[cont_cols] = scaler.fit_transform(X_train[cont_cols])
X_test[cont_cols] = scaler.transform(X_test[cont_cols])

print("14. Escalonamento concluído!")
display(X_train[cont_cols].agg(['mean', 'std']).round(4))

14. Escalonamento concluído!


,PERSONCOUNT,PEDCOUNT,PEDCYLCOUNT,VEHCOUNT
mean,-0.0,-0.0,-0.0,0.0
std,1.0,1.0,1.0,1.0


## 16. Verificar shape final de X_train, X_test, y_train, y_test

In [16]:
print("15. Resumo das dimensões (Shapes):")
print("X_train:", X_train.shape)
print("X_test: ", X_test.shape)
print("y_train:", y_train.shape)
print("y_test: ", y_test.shape)

15. Resumo das dimensões (Shapes):
X_train: (89468, 44)
X_test:  (22368, 44)
y_train: (89468,)
y_test:  (22368,)


## 17. Salvar X_train, X_test, y_train, y_test em arquivos (ex: .pkl ou .csv)

In [17]:
X_train.to_csv('X_train.csv', index=False)
X_test.to_csv('X_test.csv', index=False)
y_train.to_csv('y_train.csv', index=False)
y_test.to_csv('y_test.csv', index=False)

print("17. Arquivos salvos com sucesso (.csv)! Pré-processamento concluído.")

17. Arquivos salvos com sucesso (.csv)! Pré-processamento concluído.
